# 01 · Dựng kho tri thức Kimochi (T05)
Trích món, giá, danh mục, dị ứng (suy luận) từ bộ hội thoại gốc → `data/kb/`, `data/kimochi.db`.
Bàn / đặt bàn là **mô phỏng** (`simulated=1`). Dị ứng suy luận bằng từ khóa, `allergen_verified=0` → cần quản lý xác nhận.

In [1]:
import sys, json, pandas as pd
sys.path.insert(0, 'src')
pd.set_option('display.max_colwidth', 80)

In [2]:
from kimochi import build_kb
q = build_kb.main()
{k: v for k, v in q.items() if k != 'category_counts'}

{'raw_rows': 6355,
 'rows_mapped_to_item': 4108,
 'rows_policy_general': 2247,
 'menu_items': 106,
 'items_with_price': 74,
 'items_price_conflict': 1,
 'price_conflicts': {'Khoai Tây Chiên': {53900: 1, 49000: 1}},
 'docs_menu': 106,
 'docs_policy': 545}

## Từ điển dữ liệu — bảng `menu_items`

In [3]:
menu = pd.read_csv('../data/kb/menu.csv')
print(menu.shape)
menu.drop(columns='description').head(10)

(106, 10)


,item_id,name,category,price_vnd,allergens,is_vegetarian,spicy_level,allergen_verified,n_samples
0,M001,Combo Kaki Salmon Chuka,Combo,215000.0,cá;giáp xác;gluten;nhuyễn thể;sữa,0,0,0,45
1,M002,Combo Kani Salmon,Combo,197000.0,cá;giáp xác;gluten;sữa;trứng,0,1,0,45
2,M003,Combo Maki 215K,Combo,215000.0,cá;giáp xác;gluten;sữa,0,0,0,45
3,M004,Combo Oishi 1,Combo,NaN,cá;trứng,0,0,0,54
4,M005,Combo Oishi 2,Combo,NaN,cá;trứng,0,0,0,54
5,M006,Combo Oishi 3,Combo,NaN,cá,0,0,0,54
6,M007,Combo Oishi 4,Combo,NaN,cá;nhuyễn thể;trứng,0,0,0,54
7,M008,Combo Oishi 5,Combo,NaN,cá;nhuyễn thể;trứng,0,0,0,54
8,M009,Combo Sakura Bow,Combo,NaN,cá;nhuyễn thể;trứng,0,0,0,54
9,M010,Combo Tuna Fuji,Combo,233000.0,cá;gluten;sữa;trứng,0,1,0,39


In [4]:
import sqlite3
con = sqlite3.connect('../data/kimochi.db')
for t in ['categories','menu_items','branches','tables','reservations','promotions']:
    print(t, con.execute(f'select count(*) from {t}').fetchone()[0], [r[1] for r in con.execute(f'pragma table_info({t})')])

categories 9 ['category_id', 'name']
menu_items 106 ['item_id', 'name', 'category_id', 'price_vnd', 'allergens', 'allergen_verified', 'is_vegetarian', 'spicy_level', 'is_available', 'description']
branches 2 ['branch_id', 'name', 'district', 'address', 'hotline', 'open_time', 'close_time', 'seating']
tables 24 ['table_id', 'branch_id', 'seats', 'kind', 'simulated']
reservations 0 ['reservation_id', 'branch_id', 'table_id', 'customer_name', 'phone', 'party_size', 'start_at', 'status', 'note', 'simulated']
promotions 1 ['promo_id', 'title', 'condition', 'benefit', 'valid_from', 'valid_to', 'simulated']


## Xung đột giá và món thiếu giá (phát hiện chất lượng dữ liệu)

In [5]:
print(q['price_conflicts'])
menu[menu.price_vnd.isna()][['item_id','name','category']]

{'Khoai Tây Chiên': {53900: 1, 49000: 1}}


,item_id,name,category
3,M004,Combo Oishi 1,Combo
4,M005,Combo Oishi 2,Combo
5,M006,Combo Oishi 3,Combo
6,M007,Combo Oishi 4,Combo
7,M008,Combo Oishi 5,Combo
8,M009,Combo Sakura Bow,Combo
11,M012,Combo Umai Bow 1,Combo
12,M013,Combo Umai Bow 2,Combo
13,M014,Combo Umai Bow 3,Combo
14,M015,Sushi Box 1,Combo
